# Prompt chaining

This pattern breaks down large, complex tasks into smaller and more manageable problems. It's also known as Pipeline pattern and it's similar in nature to traditional orchestration: the output generated by one prompt is passed as input to the next promt in the chain.

## Implementation with Flyte v2

This notebook reimplements the two-step prompt chain using **Flyte v2 primitives only** — no LangChain, no LangGraph. Each prompt is a typed async task; tasks call tasks directly in pure Python, giving you durability, caching, and remote execution for free.

#### LangChain vs Flyte v2 — Key Differences

| Aspect | LangChain / LangGraph | Flyte v2 |
|--------|----------------------|----------|
| **Chain definition** | LCEL `\|` operator (`chain_a \| chain_b`) | Python functions — tasks call tasks directly |
| **State passing** | Untyped dict flowing through pipeline steps | Typed function arguments (serializable, inspectable) |
| **Checkpointing** | None — restart from scratch on failure | Each task result persists; resume from last completed step |
| **Observability** | Log parsing | Per-task traces in Flyte UI |
| **Caching** | Manual setup via cache stores | `cache="auto"` on `TaskEnvironment` |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster (no plaintext in code) |
| **Execution** | In-process only | Local or remote (containers on Kubernetes) |

In [ ]:
#First, prepare your environment
!uv pip install flyte anthropic

Using Python 3.12.12 environment at: /Users/davidmirror/code/agentic-patterns-on-v2/.venv
Audited 2 packages in 16ms


### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### Configure the execution environment

Configure your connection to the Flyte cluster. This tells Flyte where to run your workflows and how to build container images.

**Configuration Options:**
- `endpoint`: Your Flyte cluster URL
- `org`: Your organization name
- `project`: Project to organize workflows
- `domain`: Environment (development, staging, production)
- `image_builder`: Use "remote" to build images on the cluster (no local Docker required)

In [ ]:
import flyte
flyte.init_from_config()


Create a secret in Flyte for the API key (only done once):

In [ ]:

!flyte create secret ANTHROPIC_API_KEY --value sk-ant-...

- Flyte requires you to define a `TaskEnvironment`, an object that encapsulates the configuration your task will use during execution, including `Secrets`.
- Flyte allows you to [tune compute resources](https://www.union.ai/docs/v2/byoc/user-guide/task-configuration/resources/) (GPU, CPU, memory, ephemeral storage) via `Resources`.   
- [Caching](https://www.union.ai/docs/v2/byoc/user-guide/task-configuration/caching/) is enabled to save cost and drive down execution times for subsequent runs by avoiding LLM calls with the same inputs. 
- Flyte enables you to declare the desired configuration and contents of the [container image](https://www.union.ai/docs/v2/byoc/user-guide/task-configuration/container-images/) and have the image built automatically.

In [ ]:
import flyte
from flyte import TaskEnvironment, Resources

env = TaskEnvironment(
    name="prompt_chaining_env",
    image=flyte.Image.from_debian_base().with_pip_packages("anthropic"),
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")]
)


The two-prompt chain is implemented as two tasks where each task will run remotely on a container (or locally in a Python process) with its own compute resources.

This example uses the `anthropic` API and Python to:  

- Prompt 1: extract technical specs from `text_input`:


In [5]:


@env.task
async def extract_specs(text_input: str) -> str:
    import anthropic
    client = anthropic.AsyncAnthropic()
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{"role": "user", "content": f"Extract the technical specifications from:\n\n{text_input}"}],
    )
    return response.content[0].text


- Prompt 2: transform those specs into a JSON-like structure.

In [6]:
@env.task
async def specs_to_json(specifications: str) -> str:
    import anthropic
    client = anthropic.AsyncAnthropic()
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{"role": "user", "content": f"Transform into JSON with 'cpu', 'memory', 'storage':\n\n{specifications}"}],
    )
    return response.content[0].text


In Flyte V2 you can have tasks calling other tasks, giving you the flexibility to write orchestration logic in pure Python:

In [7]:
@env.task
async def process_description(text_input: str) -> str:
    specs = await extract_specs(text_input)
    return await specs_to_json(specs)

Finally, run the main task remotely:

In [ ]:
flyte.init_from_config()
run = flyte.run(process_description, text_input="This is a description of a product.")
run.wait()
print(run.outputs()[0])


**Rule of thumb:** Use this pattern when a task is too complex for a single prompt, involves multiple distinct processing stages, requires interaction with external tools between steps, or when building Agentic systems that need to perform multi-step reasoning and maintain state.